#### Инициализация Keras

In [1]:
import os

os.environ["KERAS_BACKEND"] = "torch"
import keras
import torch

print(keras.__version__)

random_seed = 42
device = "cpu"
keras.utils.set_random_seed(random_seed)

if torch.backends.mps.is_available():
    device = "mps"
    print("Ускорение GPU Apple Metal активно")
elif torch.cuda.is_available():
    device = "cuda"
    print("Ускорение GPU NVIDIA CUDA активно")
else:
    try:
        import torch_directml

        device = torch_directml.device()
        print("Ускорение GPU AMD DirectML активно")
    except ImportError:
        print("Используется CPU")

3.14.0
Ускорение GPU Apple Metal активно


#### Загрузка набора данных для задачи классификации

В данном примере используется фрагмент набора  данных Cats and Dogs Classification Dataset

В наборе данных два класса (всего 24 998 изображений): кошки (12 499 изображения) и собаки (12 499 изображения)

Ссылка: https://www.kaggle.com/datasets/bhavikjikadara/dog-and-cat-classification-dataset

In [2]:
import os

import kagglehub

path = kagglehub.dataset_download("bhavikjikadara/dog-and-cat-classification-dataset")
path = os.path.join(path, "PetImages")

#### Формирование выборок

Формируется две выборки: обучающая и валидационная (80 / 20).

В каждой выборке изображения масштабируются до размера 224 на 224 пиксела с RGB пространством.

Изображения подгружаются с диска в процессе обучения и валидации модели.

In [3]:
import keras

batch_size = 64
target_size = (224, 224)

# Загрузка обучающей выборки
train = keras.utils.image_dataset_from_directory(
    directory=path,
    validation_split=0.2,
    subset="training",
    seed=random_seed,
    image_size=target_size,
    batch_size=batch_size,
    color_mode="rgb",
    label_mode="binary",
    shuffle=True,
)

# Загрузка валидационной выборки
valid = keras.utils.image_dataset_from_directory(
    directory=path,
    validation_split=0.2,
    subset="validation",
    seed=random_seed,
    image_size=target_size,
    batch_size=batch_size,
    color_mode="rgb",
    label_mode="binary",
    shuffle=True,
)

class_names = train.class_names  # type: ignore
print(f"Имена классов: {class_names}")

Found 24998 files belonging to 2 classes.
Using 19999 files for training.
Found 24998 files belonging to 2 classes.
Using 4999 files for validation.
Имена классов: ['Cat', 'Dog']


### Пример переноса обучения с использованием предобученной модели VGGNet19

Загрузка предобученной модели VGG19:
- Загрузка весов, полученных при обучении модели на наборе данных ImageNet
- Отключение полносвязанных слоев для адаптации к новой задаче
- Модель будет работать с изображениями 224 на 224 пиксела и RGB пространством

In [4]:
from keras.applications.vgg19 import VGG19

vgg19 = VGG19(include_top=False, weights="imagenet", input_shape=(224, 224, 3), pooling=None)

vgg19.trainable = False

80134624/80134624 ━━━━━━━━━━━━━━━━━━━━ 9s 0us/step


#### Проектирование архитектуры ИНС на основе предобученной модели

In [5]:
from keras.layers import Dense, Dropout, Flatten
from keras.models import Sequential

tl_model = Sequential()
tl_model.add(vgg19)

# Добавление собственных слоев (в них будет проводиться обучение для текущей задачи)
tl_model.add(Flatten(name="flattened"))
tl_model.add(Dropout(0.5, name="dropout"))
tl_model.add(Dense(1, activation="sigmoid", name="predictions"))

tl_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg19 (Functional)              │ (None, 7, 7, 512)      │    20,024,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flattened (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 1)              │        25,089 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,049,473 (76.48 MB)

 Trainable params: 25,089 (98.00 KB)

 Non-trainable params: 20,024,384 (76.39 MB)

#### Обучение глубокой модели

Обучение остановлено после второго шага, так как качество модели приемлемое

In [6]:
tl_model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

tl_model.fit(x=train, validation_data=valid, epochs=3)
tl_model.save("models/tl_model.keras")

Epoch 1/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 330s 1s/step - accuracy: 0.9385 - loss: 0.7704 - val_accuracy: 0.9602 - val_loss: 0.5031
Epoch 2/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 331s 1s/step - accuracy: 0.9615 - loss: 0.5213 - val_accuracy: 0.9688 - val_loss: 0.4401
Epoch 3/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 333s 1s/step - accuracy: 0.9675 - loss: 0.4620 - val_accuracy: 0.9696 - val_loss: 0.4054


#### Оценка качества модели

Качество модели чуть больше 96 %.

In [7]:
tl_model.evaluate(valid)

79/79 ━━━━━━━━━━━━━━━━━━━━ 70s 860ms/step - accuracy: 0.9696 - loss: 0.4054


[0.4053792655467987, 0.9695939421653748]